<a href="https://colab.research.google.com/github/gkienpham-cmd/flashattention-cuda/blob/main/notebooks/colab_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# flashattention-cuda — Colab bootstrap (T4)

One pass: confirm the GPU → build the kernel → predict the roofline → test vs SDPA → benchmark.

**Runtime → Change runtime type → T4 GPU** before running. Everything below runs on the GPU;
nothing here works on a CPU-only runtime.

## 0. Confirm the hardware (Step 0 of the brief)
We record GPU model, compute capability, and clocks — every benchmark row must carry these,
and the free-tier T4 thermally throttles.

In [1]:
!nvidia-smi --query-gpu=name,compute_cap,clocks.sm,clocks.max.sm,memory.total,temperature.gpu --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| capability', torch.cuda.get_device_capability())

name, compute_cap, clocks.current.sm [MHz], clocks.max.sm [MHz], memory.total [MiB], temperature.gpu
Tesla T4, 7.5, 300 MHz, 1590 MHz, 15360 MiB, 52
torch 2.11.0+cu128 | cuda 12.8 | capability (7, 5)


## 1. Get the repo
`REPO_URL` is already set to the public repo, so the clone just works. (If you fork it, point
`REPO_URL` at your fork.) The repo root is added to `sys.path` so the notebook can import it.

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works on Colab
import os, sys, subprocess
if not os.path.isdir('flashattention-cuda'):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
os.chdir('flashattention-cuda')
sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Predict the roofline BEFORE running anything
The per-step loop starts here: predict the limiter, then check it against reality below.

In [3]:
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x64 --precision fp32 --materialize-s
print()
!python -m roofline.predict --arch sm_75 --shape 1x8x2048x128 --precision fp32 --materialize-s

arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=64  precision=fp32  materialize_S=True  tile=1x1
LIMITER     : HBM   (predicted lower bound 109.065 ms)
  t_mma     :    1.060 ms   util   1.0%
  t_hbm     :  109.065 ms   util 100.0%
  t_mufu    :    0.033 ms   util   0.0%
intensity   : 0.2 FLOP/byte   (arch ridge 25.3; BELOW -> memory-bound)

arch        : Tesla T4 (sm_75)
shape       : B=1 H=8 N=2048 d=128  precision=fp32  materialize_S=True  tile=1x1
LIMITER     : HBM   (predicted lower bound 216.452 ms)
  t_mma     :    2.121 ms   util   1.0%
  t_hbm     :  216.452 ms   util 100.0%
  t_mufu    :    0.033 ms   util   0.0%
intensity   : 0.2 FLOP/byte   (arch ridge 25.3; BELOW -> memory-bound)


## 3. Build the v1 kernel (JIT)
First call compiles with nvcc (~1 min); cached afterwards. `verbose=True` prints the build.

In [4]:
# Clean build: a failed compile (e.g. ninja missing on the first try) leaves a stale cache dir
# with a version stamp but no .so, so torch SKIPS the rebuild and then fails to import. Remove
# any partial fa_* build dir (one with no compiled .so); a good cached build is kept, so re-runs
# stay fast.
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_*')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True)
        print('cleaned stale build:', d)
print('clean-build check done')

clean-build check done


In [5]:
# cpp_extension.load() compiles via ninja, which Colab doesn't always ship — install it first.
!pip install -q ninja
from bindings.load import build_kernel
mod = build_kernel('v1_naive')
print('built:', mod)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.2 MB/s eta 0:00:00
built: <module 'fa_v1_naive' from '/root/.cache/torch_extensions/py312_cu128/fa_v1_naive/fa_v1_naive.so'>


## 4. Correctness vs SDPA (documented tolerance: atol/rtol 1e-4)

In [6]:
!python -m pytest tests/ -q

.........                                                                [100%]
9 passed in 5.12s


## 5. Benchmark vs SDPA across the sweep
Expect to be **slower than SDPA** — SDPA is already a fused efficient kernel. This is the
'before'. Paste the numbers into `docs/results.md` and compare to the roofline prediction.

In [7]:
!python -m bench.harness --backend v1_naive --precision fp32

# device: Tesla T4 (sm_75)  clock~0MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
ninja: no work to do.
        1x8x512x64 |   6.765/ 14.166 |   0.201/  5.097 |    0.03x |    6.055e+05 | HBM (~6.82ms)
       1x8x512x128 |  10.079/ 10.181 |   0.332/  0.374 |    0.03x |    4.064e+05 | HBM (~13.53ms)
       1x8x2048x64 |  85.563/ 89.470 |   3.029/  3.215 |    0.04x |    1.915e+05 | HBM (~109.07ms)
      1x8x2048x128 | 166.589/171.395 |   5.561/  6.263 |    0.03x |    9.835e+04 | HBM (~216.45ms)
       1x8x8192x64 | 1763.709/1816.840 |  47.998/ 48.962 |    0.03x |    3.716e+04 | HBM (~1744.88ms)
      1x8x8192x128 | 3112.675/3134.220 |  96.435/ 97.050 |    0.03x |    2.105e+04 | HBM (~3462.92ms)


## 6. (Optional) Nsight Compute capture
Confirms the limiter: DRAM throughput near peak at d=64 (the bandwidth wall), low MMA/MUFU.
`ncu` may need a GPU runtime that allows profiling; see profiling/GUIDE.md.

In [8]:
!bash profiling/capture.sh v1_naive || echo 'ncu unavailable on this runtime; read GUIDE.md'

==PROF== Connected to process 2680 (/usr/bin/python3.12)
# device: Tesla T4 (sm_75)  clock~0MHz  backend=v1_naive  precision=fp32  causal=False
#            shape |    ours p50/p99 ms |    sdpa p50/p99 ms |  speedup |  tok/s(ours) | roofline
==PROF== Profiling "distribution_elementwise_grid..." - 0 (1/3): 0%....50%....100% - 31 passes
==PROF== Profiling "distribution_elementwise_grid..." - 1 (2/3): 0%....50%....100% - 31 passes
==PROF== Profiling "distribution_elementwise_grid..." - 2 (3/3): 0%....50%....100% - 31 passes
ninja: no work to do.
        1x8x512x64 |   5.640/  9.332 |   0.310/  0.494 |    0.06x |    7.262e+05 | HBM (~6.82ms)
       1x8x512x128 |  10.658/ 10.762 |   0.513/  1.286 |    0.05x |    3.843e+05 | HBM (~13.53ms)
       1x8x2048x64 |  90.671/ 93.425 |   3.313/  3.730 |    0.04x |    1.807e+05 | HBM (~109.07ms)
      1x8x2048x128 | 177.176/180.996 |   6.224/  7.191 |    0.04x |    9.247e+04 | HBM (~216.45ms)
       1x8x8192x64 | 1675.388/1785.515 |  49.334/ 50.263 |